In [1]:
# Setup
import os, sys, torch
from argparse import Namespace

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
os.chdir(repo_root)
sys.path.append(repo_root) if repo_root not in sys.path else None

from revlm.config_utils import configure_args
from revlm import VQAModel, VQADataset
from revlm.editors.auto_q import BiasLayer


In [2]:
# Load model & dataset
model_name = "qwen3_4b"  # Options: "blip", "llava", "qwen3", "qwen3_4b"

args = Namespace(config="revlm/config/config.yaml", editor="ike_chain", model_name=model_name,
                 dataset_name="aokvqa", task="mc", batch_size=1, split="all",
                 rationale=False, cot=False, subsample=100, overwrite=False)
args.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
config = configure_args(args, config_path=args.config)

model = VQAModel(config)
dataset = VQADataset(config)
print(f"Model: {config.model.name}, Dataset: {len(dataset)} samples")


Task evaluation metrics will be saved to results/te/ike_chain/Qwen3-VL-4B-Instruct/aokvqa
Edit evaluation metrics will be saved to results/ee/ike_chain/Qwen3-VL-4B-Instruct/aokvqa
Predictions will be saved to results/pred/Qwen3-VL-4B-Instruct/aokvqa
Post-edit predictions will be saved to results/pred_postedit/ike_chain/Qwen3-VL-4B-Instruct/aokvqa
Unified filename to save: mc_all_sub100.json
Model: Qwen/Qwen3-VL-4B-Instruct, Dataset: 18195 samples


In [3]:
# Initialize BiasLayer and get candidate layers
bias = BiasLayer(config, model, pool_method="last")
layers = bias.get_candidate_layers()


[BiasLayer] 92 layers (vision: 48, merger: 8, language: 36)


In [ ]:
# Compute bias at each layer
# n_samples × 9 forwards per layer (1 anchor + 2 i+ + 2 t+ + 2 i- + 2 t-)
scores = bias.compute(dataset, layers, n_samples=3)


[BiasLayer] Computing bias at 92 layers, 2 samples


layer_bias:   0%|                                                     | 0/92 [00:00<?, ?it/s]

[Augmenter] Loading Qwen/Qwen2.5-1.5B-Instruct...
[BiasViz] Edit 0: 9 embeddings computed


layer_bias:   1%|▍                                          | 1/92 [00:51<1:17:51, 51.33s/it]

[BiasViz] Edit 1: 9 embeddings computed
  model.visual.blocks.0.mlp.linear_fc1.weight: vis=+17.464, txt=-20.502
[Augmenter] Loading Qwen/Qwen2.5-1.5B-Instruct...


In [ ]:
# Plot bias scores
# vision_bias (green): positive = image representation too weak
# text_bias (blue): positive = text representation too weak
bias.plot(scores)


In [ ]:
# Cleanup
bias.cleanup()
del model
torch.cuda.empty_cache()


# Load Saved Results (After Running Jobs)

After running `jobs/bias_layer/run.sh`, use the cells below to load aggregated results with error bars.


In [ ]:
# Setup for loading saved results
import os, sys, torch
from argparse import Namespace

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
os.chdir(repo_root)
sys.path.append(repo_root) if repo_root not in sys.path else None

from revlm.config_utils import configure_args
from revlm.editors.auto_q import BiasLayer


In [ ]:
# Load aggregated results from saved runs (with error bars)
model_name = "qwen3_4b"  # Options: "blip", "llava", "qwen3", "qwen3_4b"

args = Namespace(config="revlm/config/config.yaml", editor="ike_chain", model_name=model_name,
                 dataset_name="aokvqa", task="mc", batch_size=1, split="all",
                 rationale=False, cot=False)
args.device = torch.device("cpu")
config = configure_args(args, config_path=args.config)

# Create BiasLayer just for loading (no model needed)
bias_loader = BiasLayer.__new__(BiasLayer)
bias_loader.config = config
bias_loader.device = args.device

# Load aggregated results
agg_scores = bias_loader.load_results_k(out_dir="results/bias_layer")
if agg_scores:
    print(f"Loaded {len(agg_scores)} layers")


In [ ]:
# Plot aggregated results with error bars
if agg_scores:
    bias_loader._classify_layers = lambda layers: BiasLayer._classify_layers(bias_loader, layers)
    bias_loader.plot = lambda scores, **kwargs: BiasLayer.plot(bias_loader, scores, **kwargs)
    bias_loader.plot(agg_scores)
else:
    print("No saved results found. Run jobs/bias_layer/run.sh first.")
